In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
doctors_df = spark.read.option("header","true") \
    .option("inferSchema","true") \
    .csv("dbfs:/hospital_data/doctors.csv")

visits_df = spark.read.option("header","true") \
    .option("inferSchema","true") \
    .csv("dbfs:/hospital_data/visits.csv")

hospital_df = spark.read.option("multiline","true") \
    .json("dbfs:/hospital_data/hospital_config.json")

In [0]:
doctors_df.show()
visits_df.show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|          1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|          1100|
+---------+-----------+--------------+---------+----------------+--------------+

+--------+------------+----

In [0]:
doctors_df.printSchema()
visits_df.printSchema()

root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_f: integer (nullable = true)

root
 |-- visit_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = true)
 |-- payment_: string (nullable = true)



In [0]:
doctors_df.count()

8

In [0]:
visits_df.count()

10

In [0]:
doctors_df.filter(col("city")=="Hyderabad").show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
doctors_df.filter(col("specialization")=="Cardiology").show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
doctors_df.filter(col("experience_years") > 10).show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
visits_df.filter(col("bill_amount") > 5000).show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
+--------+------------+---------+----------+-------------+-----------+--------+



In [0]:
visits_df.filter(col("payment_")=="Pending").show()

+--------+------------+---------+----------+------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+------------+-----------+--------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|       2000| Pending|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL| Pending|
+--------+------------+---------+----------+------------+-----------+--------+



In [0]:
visits_df.filter(col("payment_")=="Paid").show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
+--------+------------+---------+----------+-------------+-----------+--------+



In [0]:
doctors_df.groupBy("specialization") \
          .agg(avg("consultation_f")) \
          .show()

+--------------+-------------------+
|specialization|avg(consultation_f)|
+--------------+-------------------+
|     Neurology|             1900.0|
|   Dermatology|             1050.0|
|    Cardiology|             2250.0|
|    Pediatrics|             1200.0|
|   Orthopedics|             2500.0|
+--------------+-------------------+



In [0]:
doctors_df.groupBy("specialization") \
          .agg(max("consultation_f")) \
          .show()

+--------------+-------------------+
|specialization|max(consultation_f)|
+--------------+-------------------+
|     Neurology|               2000|
|   Dermatology|               1100|
|    Cardiology|               3000|
|    Pediatrics|               1200|
|   Orthopedics|               2500|
+--------------+-------------------+



In [0]:
doctors_df.groupBy("city") \
          .count() \
          .show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [0]:
doctors_df.groupBy("specialization") \
          .count() \
          .show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|     Neurology|    2|
|   Dermatology|    2|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Orthopedics|    1|
+--------------+-----+



In [0]:
visits_df.agg(sum("bill_amount")) \
         .show()

+----------------+
|sum(bill_amount)|
+----------------+
|           46500|
+----------------+



In [0]:
visits_df.agg(avg("bill_amount")) \
         .show()

+-----------------+
| avg(bill_amount)|
+-----------------+
|5166.666666666667|
+-----------------+



In [0]:
visits_df.agg(max("bill_amount")) \
         .show()

+----------------+
|max(bill_amount)|
+----------------+
|           12000|
+----------------+



In [0]:
visits_df.agg(min("bill_amount")) \
         .show()


+----------------+
|min(bill_amount)|
+----------------+
|            1500|
+----------------+



In [0]:
doctors_df.orderBy(desc("consultation_f")) \
          .show()


+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|          1800|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|          1100|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
visits_df.orderBy(desc("bill_amount")) \
         .show()


+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid|
|   V1008|  Meera Nair|     D103|2026-01

In [0]:
visits_df.filter(col("bill_amount").isNull()) \
         .show()

+--------+------------+---------+----------+------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+------------+-----------+--------+
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL| Pending|
+--------+------------+---------+----------+------------+-----------+--------+



In [0]:
visits_df = visits_df.fillna({
    "bill_amount":0
})
visits_df.show()

+--------+------------+---------+----------+-------------+-----------+--------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+--------+------------+---------+----------+-------------+-----------+--------+-----+----------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|600.0|   12600.0|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid| 75.0|    1575.0|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|350.0|    7350.0|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|200.0|    4200.0|
|   V1008|  Meera Nair|     D1

In [0]:
visits_df = visits_df.withColumn(
    "tax",
    col("bill_amount") * 0.05
)
visits_df.show()

+--------+------------+---------+----------+-------------+-----------+--------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+--------+------------+---------+----------+-------------+-----------+--------+-----+----------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|600.0|   12600.0|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid| 75.0|    1575.0|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|350.0|    7350.0|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|200.0|    4200.0|
|   V1008|  Meera Nair|     D1

In [0]:
visits_df = visits_df.withColumn(
    "final_bill",
    col("bill_amount") + col("tax")
)

visits_df.show()

+--------+------------+---------+----------+-------------+-----------+--------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+--------+------------+---------+----------+-------------+-----------+--------+-----+----------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|600.0|   12600.0|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid| 75.0|    1575.0|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|350.0|    7350.0|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|200.0|    4200.0|
|   V1008|  Meera Nair|     D1

In [0]:
joined_df = doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
)

joined_df.show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   

In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "left"
).show()


+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|   V1005|  Farhan Ali|2026-01-12|        Fever|       1500|    Paid| 75.0|    1575.0|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|   V1004| Sneha Patel|2026-01-11|     Fracture|      12000|    Paid|600.0|   12600.0|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|   V1009|   Kiran Rao|2026-01-14|    Back Pain|       6000|    Paid|300.0|    6300.0|
|   

In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "right"
).show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   

In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "full"
).show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|   V1005|  Farhan Ali|2026-01-12|        Fever|       1500|    Paid| 75.0|    1575.0|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|   V1004| Sneha Patel|2026-01-11|     Fracture|      12000|    Paid|600.0|   12600.0|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|   V1009|   Kiran Rao|2026-01-14|    Back Pain|       6000|    Paid|300.0|    6300.0|
|   

In [0]:
visits_df.join(
    doctors_df,
    "doctor_id",
    "left_anti"
).show()

+---------+--------+------------+----------+---------+-----------+--------+-----+----------+
|doctor_id|visit_id|patient_name|visit_date|diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+--------+------------+----------+---------+-----------+--------+-----+----------+
|     D120|   V1007| Arjun Verma|2026-01-13| Migraine|       4000|    Paid|200.0|    4200.0|
+---------+--------+------------+----------+---------+-----------+--------+-----+----------+



In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "left_anti"
).show()

+---------+-----------+--------------+-----+----------------+--------------+
|doctor_id|doctor_name|specialization| city|experience_years|consultation_f|
+---------+-----------+--------------+-----+----------------+--------------+
|     D107| Dr. Farhan|     Neurology| Pune|               5|          1800|
|     D108|  Dr. Nisha|   Dermatology|Kochi|               9|          1100|
+---------+-----------+--------------+-----+----------------+--------------+



In [0]:
joined_df.groupBy(
    "doctor_name"
).count().show()

+-----------+-----+
|doctor_name|count|
+-----------+-----+
|  Dr. Meera|    1|
|  Dr. Priya|    1|
| Dr. Ramesh|    2|
| Dr. Suresh|    2|
|  Dr. Anita|    2|
|  Dr. Kiran|    1|
+-----------+-----+



In [0]:
joined_df.groupBy(
    "doctor_name"
).agg(
    sum("bill_amount").alias("revenue")
).show()

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
|  Dr. Meera|   1500|
|  Dr. Priya|   3500|
| Dr. Ramesh|  10500|
| Dr. Suresh|  18000|
|  Dr. Anita|   2000|
|  Dr. Kiran|   7000|
+-----------+-------+



In [0]:
joined_df.groupBy(
    "doctor_name"
).agg(
    sum("bill_amount").alias("revenue")
).orderBy(
    desc("revenue")
).show(1)

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
| Dr. Suresh|  18000|
+-----------+-------+
only showing top 1 row


In [0]:
joined_df.groupBy(
    "specialization"
).agg(
    sum("bill_amount").alias("revenue")
).orderBy(
    desc("revenue")
).show(1)

+--------------+-------+
|specialization|revenue|
+--------------+-------+
|   Orthopedics|  18000|
+--------------+-------+
only showing top 1 row


In [0]:
joined_df.groupBy(
    "specialization"
).agg(
    avg("bill_amount").alias("average_revenue")
).show()

+--------------+-----------------+
|specialization|  average_revenue|
+--------------+-----------------+
|     Neurology|           3500.0|
|   Dermatology|           1000.0|
|    Cardiology|5833.333333333333|
|    Pediatrics|           1500.0|
|   Orthopedics|           9000.0|
+--------------+-----------------+



In [0]:
joined_df.groupBy(
    "city"
).agg(
    sum("bill_amount").alias("total_revenue")
).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         3500|
|  Chennai|         2000|
|   Mumbai|        18000|
|    Delhi|         1500|
|Hyderabad|        17500|
+---------+-------------+



In [0]:
joined_df.groupBy(
    "doctor_name"
).count().show()

+-----------+-----+
|doctor_name|count|
+-----------+-----+
|  Dr. Meera|    1|
|  Dr. Priya|    1|
| Dr. Ramesh|    2|
| Dr. Suresh|    2|
|  Dr. Anita|    2|
|  Dr. Kiran|    1|
+-----------+-----+



In [0]:
joined_df.groupBy(
    "doctor_name"
).agg(
    sum("bill_amount").alias("revenue")
).orderBy(
    desc("revenue")
).show(3)

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
| Dr. Suresh|  18000|
| Dr. Ramesh|  10500|
|  Dr. Kiran|   7000|
+-----------+-------+
only showing top 3 rows


In [0]:
doctor_performance = joined_df.select(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city",
    "patient_name",
    "bill_amount",
    "payment_"
)

doctor_performance.show()

+---------+-----------+--------------+---------+------------+-----------+--------+
|doctor_id|doctor_name|specialization|     city|patient_name|bill_amount|payment_|
+---------+-----------+--------------+---------+------------+-----------+--------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|Rahul Sharma|       5000|    Paid|
|     D102|  Dr. Priya|     Neurology|Bangalore| Priya Reddy|       3500|    Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|  Amit Kumar|       2000| Pending|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai| Sneha Patel|      12000|    Paid|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|  Farhan Ali|       1500|    Paid|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|  Neha Singh|       7000|    Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|  Meera Nair|          0| Pending|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|   Kiran Rao|       6000|    Paid|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad| Nisha Reddy|       5500|    Paid|
+---

In [0]:
hospital_df.show()
hospital_df.printSchema()

+---------+--------------------+-----------+----------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|
+---------+--------------------+-----------+----------------+--------------------+
|Hyderabad|{apollo@mail.com,...|       H101| Apollo Hospital|[Cardiology, Neur...|
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|
+---------+--------------------+-----------+----------------+--------------------+

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
hospital_flat = hospital_df.select(
    "hospital_id",
    "hospital_name",
    "city",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    "services"
)

In [0]:
hospital_flat.select(
    "phone"
).show()

+----------+
|     phone|
+----------+
|9876500011|
|      NULL|
|9876500013|
+----------+



In [0]:
hospital_flat.select(
    "email"
).show()

+----------------+
|           email|
+----------------+
| apollo@mail.com|
|yashoda@mail.com|
|            NULL|
+----------------+



In [0]:
hospital_flat.filter(
    col("phone").isNull()
).show()

+-----------+----------------+---------+-----+----------------+--------------------+
|hospital_id|   hospital_name|     city|phone|           email|            services|
+-----------+----------------+---------+-----+----------------+--------------------+
|       H102|Yashoda Hospital|Hyderabad| NULL|yashoda@mail.com|[Cardiology, Orth...|
+-----------+----------------+---------+-----+----------------+--------------------+



In [0]:
hospital_flat.filter(
    col("email").isNull()
).show()

+-----------+-------------+---------+----------+-----+--------------------+
|hospital_id|hospital_name|     city|     phone|email|            services|
+-----------+-------------+---------+----------+-----+--------------------+
|       H103|Care Hospital|Bangalore|9876500013| NULL|[Neurology, Pedia...|
+-----------+-------------+---------+----------+-----+--------------------+



In [0]:
hospital_flat = hospital_flat.fillna({
    "phone":"Not Available"
})

In [0]:
hospital_flat = hospital_flat.fillna({
    "email":"Not Available"
})


In [0]:
hospital_flat.select(
    "hospital_name",
    "city"
).show()

+----------------+---------+
|   hospital_name|     city|
+----------------+---------+
| Apollo Hospital|Hyderabad|
|Yashoda Hospital|Hyderabad|
|   Care Hospital|Bangalore|
+----------------+---------+



In [0]:
hospital_flat.select(
    "hospital_name",
    "phone"
).show()

+----------------+-------------+
|   hospital_name|        phone|
+----------------+-------------+
| Apollo Hospital|   9876500011|
|Yashoda Hospital|Not Available|
|   Care Hospital|   9876500013|
+----------------+-------------+



In [0]:
hospital_services = hospital_flat.withColumn(
    "service",
    explode("services")
)

hospital_services.show()

+-----------+----------------+---------+-------------+----------------+--------------------+-----------+
|hospital_id|   hospital_name|     city|        phone|           email|            services|    service|
+-----------+----------------+---------+-------------+----------------+--------------------+-----------+
|       H101| Apollo Hospital|Hyderabad|   9876500011| apollo@mail.com|[Cardiology, Neur...| Cardiology|
|       H101| Apollo Hospital|Hyderabad|   9876500011| apollo@mail.com|[Cardiology, Neur...|  Neurology|
|       H101| Apollo Hospital|Hyderabad|   9876500011| apollo@mail.com|[Cardiology, Neur...|Dermatology|
|       H102|Yashoda Hospital|Hyderabad|Not Available|yashoda@mail.com|[Cardiology, Orth...| Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Not Available|yashoda@mail.com|[Cardiology, Orth...|Orthopedics|
|       H103|   Care Hospital|Bangalore|   9876500013|   Not Available|[Neurology, Pedia...|  Neurology|
|       H103|   Care Hospital|Bangalore|   9876500013| 

In [0]:
hospital_services.select(
    "hospital_name",
    "service"
).show()

+----------------+-----------+
|   hospital_name|    service|
+----------------+-----------+
| Apollo Hospital| Cardiology|
| Apollo Hospital|  Neurology|
| Apollo Hospital|Dermatology|
|Yashoda Hospital| Cardiology|
|Yashoda Hospital|Orthopedics|
|   Care Hospital|  Neurology|
|   Care Hospital| Pediatrics|
+----------------+-----------+



In [0]:
hospital_flat.groupBy(
    "city"
).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|Hyderabad|    2|
+---------+-----+



In [0]:
hospital_services.groupBy(
    "service"
).count().show()

+-----------+-----+
|    service|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    1|
| Cardiology|    2|
| Pediatrics|    1|
|Orthopedics|    1|
+-----------+-----+



In [0]:
hospital_services.filter(
    col("service") == "Cardiology"
).show()

+-----------+----------------+---------+-------------+----------------+--------------------+----------+
|hospital_id|   hospital_name|     city|        phone|           email|            services|   service|
+-----------+----------------+---------+-------------+----------------+--------------------+----------+
|       H101| Apollo Hospital|Hyderabad|   9876500011| apollo@mail.com|[Cardiology, Neur...|Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Not Available|yashoda@mail.com|[Cardiology, Orth...|Cardiology|
+-----------+----------------+---------+-------------+----------------+--------------------+----------+



In [0]:
hospital_services.filter(
    col("service") == "Neurology"
).show()

+-----------+---------------+---------+----------+---------------+--------------------+---------+
|hospital_id|  hospital_name|     city|     phone|          email|            services|  service|
+-----------+---------------+---------+----------+---------------+--------------------+---------+
|       H101|Apollo Hospital|Hyderabad|9876500011|apollo@mail.com|[Cardiology, Neur...|Neurology|
|       H103|  Care Hospital|Bangalore|9876500013|  Not Available|[Neurology, Pedia...|Neurology|
+-----------+---------------+---------+----------+---------------+--------------------+---------+



In [0]:
hospital_services.filter(
    col("service") == "Orthopedics"
).show()


+-----------+----------------+---------+-------------+----------------+--------------------+-----------+
|hospital_id|   hospital_name|     city|        phone|           email|            services|    service|
+-----------+----------------+---------+-------------+----------------+--------------------+-----------+
|       H102|Yashoda Hospital|Hyderabad|Not Available|yashoda@mail.com|[Cardiology, Orth...|Orthopedics|
+-----------+----------------+---------+-------------+----------------+--------------------+-----------+



In [0]:
hospital_services.filter(
    col("service") == "Pediatrics"
).show()


+-----------+-------------+---------+----------+-------------+--------------------+----------+
|hospital_id|hospital_name|     city|     phone|        email|            services|   service|
+-----------+-------------+---------+----------+-------------+--------------------+----------+
|       H103|Care Hospital|Bangalore|9876500013|Not Available|[Neurology, Pedia...|Pediatrics|
+-----------+-------------+---------+----------+-------------+--------------------+----------+



In [0]:
hospital_flat.write.mode(
    "overwrite"
).parquet(
    "hospital_parquet"
)


In [0]:
hospital_flat.withColumn(
    "services",
    concat_ws("|", col("services"))
).write.mode(
    "overwrite"
).option(
    "header",
    "true"
).csv(
    "hospital_csv"
)

In [0]:
doctor_revenue = joined_df.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    sum("bill_amount").alias("revenue")
)

In [0]:
w = Window.orderBy(
    desc("revenue")
)

doctor_revenue.withColumn(
    "rank",
    rank().over(w)
).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   6|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctor_revenue.withColumn(
    "dense_rank",
    dense_rank().over(w)
).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|dense_rank|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|         6|
+---------+-----------+--------------+---------+-------+----------+



In [0]:
doctor_revenue.withColumn(
    "row_number",
    row_number().over(w)
).show()


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|row_number|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|         6|
+---------+-----------+--------------+---------+-------+----------+



In [0]:
doctor_revenue.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank") == 1
).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+------+-------+----+
|doctor_id|doctor_name|specialization|  city|revenue|rank|
+---------+-----------+--------------+------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|Mumbai|  18000|   1|
+---------+-----------+--------------+------+-------+----+



In [0]:
doctor_revenue.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank") <= 3
).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
+---------+-----------+--------------+---------+-------+----+



In [0]:
w = Window.partitionBy(
    "specialization"
).orderBy(
    desc("revenue")
)


In [0]:
doctor_revenue.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank") == 1
).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctor_revenue.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank") <= 2
).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   2|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
w = Window.orderBy(
    desc("revenue")
)

doctor_revenue.withColumn(
    "running_revenue",
    sum("revenue").over(w)
).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+---------------+
|doctor_id|doctor_name|specialization|     city|revenue|running_revenue|
+---------+-----------+--------------+---------+-------+---------------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|          18000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|          28500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|          35500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|          39000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|          41000|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|          42500|
+---------+-----------+--------------+---------+-------+---------------+



In [0]:
doctor_revenue.withColumn(
    "previous_revenue",
    lag("revenue").over(w)
).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------------+
|doctor_id|doctor_name|specialization|     city|revenue|previous_revenue|
+---------+-----------+--------------+---------+-------+----------------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|            NULL|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|           18000|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|           10500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|            7000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|            3500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|            2000|
+---------+-----------+--------------+---------+-------+----------------+



In [0]:
doctor_revenue.withColumn(
    "next_revenue",
    lead("revenue").over(w)
).show()


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+------------+
|doctor_id|doctor_name|specialization|     city|revenue|next_revenue|
+---------+-----------+--------------+---------+-------+------------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|       10500|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|        7000|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|        3500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|        2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|        1500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|        NULL|
+---------+-----------+--------------+---------+-------+------------+



In [0]:
doctor_revenue.withColumn(
    "previous_revenue",
    lag("revenue").over(w)
).withColumn(
    "difference",
    col("revenue") - col("previous_revenue")
).show()


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|previous_revenue|difference|
+---------+-----------+--------------+---------+-------+----------------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|            NULL|      NULL|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|           18000|     -7500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|           10500|     -3500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|            7000|     -3500|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|            3500|     -1500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|            2000|      -500|
+---------+-----------+--------------+---------+-------+----------------+----------+



In [0]:
doctor_revenue.withColumn(
    "next_revenue",
    lead("revenue").over(w)
).withColumn(
    "difference",
    col("next_revenue") - col("revenue")
).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+------------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|next_revenue|difference|
+---------+-----------+--------------+---------+-------+------------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|       10500|     -7500|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|        7000|     -3500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|        3500|     -3500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|        2000|     -1500|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|        1500|      -500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|        NULL|      NULL|
+---------+-----------+--------------+---------+-------+------------+----------+



In [0]:
w = Window.partitionBy(
    "city"
).orderBy(
    desc("revenue")
)

doctor_revenue.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank") == 1
).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
w = Window.partitionBy(
    "city"
).orderBy(
    col("revenue")
)

doctor_revenue.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank") == 1
).show()


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctor_revenue.withColumn(
    "rank",
    rank().over(
        Window.orderBy(desc("revenue"))
    )
).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   6|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctors_df.createOrReplaceTempView("doctors")
visits_df.createOrReplaceTempView("visits")
hospital_services.createOrReplaceTempView("hospitals")

In [0]:
spark.sql("select * from doctors").show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|          1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|          1100|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
spark.sql("""
select specialization,
count(*) as total_doctors
from doctors
group by specialization
""").show()

+--------------+-------------+
|specialization|total_doctors|
+--------------+-------------+
|     Neurology|            2|
|   Dermatology|            2|
|    Cardiology|            2|
|    Pediatrics|            1|
|   Orthopedics|            1|
+--------------+-------------+



In [0]:
spark.sql("""
select city,
count(*) as total_doctors
from doctors
group by city
""").show()


+---------+-------------+
|     city|total_doctors|
+---------+-------------+
|Bangalore|            1|
|    Kochi|            1|
|  Chennai|            1|
|   Mumbai|            1|
|     Pune|            1|
|    Delhi|            1|
|Hyderabad|            2|
+---------+-------------+



In [0]:
spark.sql("""
select d.doctor_name,
sum(v.bill_amount) as revenue
from doctors d
join visits v
on d.doctor_id=v.doctor_id
group by d.doctor_name
""").show()

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
|  Dr. Meera|   1500|
|  Dr. Priya|   3500|
| Dr. Ramesh|  10500|
| Dr. Suresh|  18000|
|  Dr. Anita|   2000|
|  Dr. Kiran|   7000|
+-----------+-------+



In [0]:
spark.sql("""
select d.specialization,
sum(v.bill_amount) as revenue
from doctors d
join visits v
on d.doctor_id=v.doctor_id
group by d.specialization
""").show()

+--------------+-------+
|specialization|revenue|
+--------------+-------+
|     Neurology|   3500|
|   Dermatology|   2000|
|    Cardiology|  17500|
|    Pediatrics|   1500|
|   Orthopedics|  18000|
+--------------+-------+



In [0]:
spark.sql("""
select d.doctor_name,
sum(v.bill_amount) as revenue
from doctors d
join visits v
on d.doctor_id=v.doctor_id
group by d.doctor_name
order by revenue desc
limit 5
""").show()


+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
| Dr. Suresh|  18000|
| Dr. Ramesh|  10500|
|  Dr. Kiran|   7000|
|  Dr. Priya|   3500|
|  Dr. Anita|   2000|
+-----------+-------+



In [0]:
spark.sql("""
select *
from visits
where payment_='Pending'
""").show()

+--------+------------+---------+----------+------------+-----------+--------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|  tax|final_bill|
+--------+------------+---------+----------+------------+-----------+--------+-----+----------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|       2000| Pending|100.0|    2100.0|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|          0| Pending|  0.0|       0.0|
+--------+------------+---------+----------+------------+-----------+--------+-----+----------+



In [0]:
spark.sql("""
select *
from hospitals
where service='Cardiology'
""").show()

+-----------+----------------+---------+-------------+----------------+--------------------+----------+
|hospital_id|   hospital_name|     city|        phone|           email|            services|   service|
+-----------+----------------+---------+-------------+----------------+--------------------+----------+
|       H101| Apollo Hospital|Hyderabad|   9876500011| apollo@mail.com|[Cardiology, Neur...|Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Not Available|yashoda@mail.com|[Cardiology, Orth...|Cardiology|
+-----------+----------------+---------+-------------+----------------+--------------------+----------+



In [0]:
spark.sql("""
select *
from hospitals
where service='Neurology'
""").show()

+-----------+---------------+---------+----------+---------------+--------------------+---------+
|hospital_id|  hospital_name|     city|     phone|          email|            services|  service|
+-----------+---------------+---------+----------+---------------+--------------------+---------+
|       H101|Apollo Hospital|Hyderabad|9876500011|apollo@mail.com|[Cardiology, Neur...|Neurology|
|       H103|  Care Hospital|Bangalore|9876500013|  Not Available|[Neurology, Pedia...|Neurology|
+-----------+---------------+---------+----------+---------------+--------------------+---------+



In [0]:
spark.sql("""
select *
from hospitals
where phone='Not Available'
or email='Not Available'
""").show()

+-----------+----------------+---------+-------------+----------------+--------------------+-----------+
|hospital_id|   hospital_name|     city|        phone|           email|            services|    service|
+-----------+----------------+---------+-------------+----------------+--------------------+-----------+
|       H102|Yashoda Hospital|Hyderabad|Not Available|yashoda@mail.com|[Cardiology, Orth...| Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Not Available|yashoda@mail.com|[Cardiology, Orth...|Orthopedics|
|       H103|   Care Hospital|Bangalore|   9876500013|   Not Available|[Neurology, Pedia...|  Neurology|
|       H103|   Care Hospital|Bangalore|   9876500013|   Not Available|[Neurology, Pedia...| Pediatrics|
+-----------+----------------+---------+-------------+----------------+--------------------+-----------+



In [0]:
spark.sql("""
select avg(consultation_f)
from doctors
""").show()


+-------------------+
|avg(consultation_f)|
+-------------------+
|             1762.5|
+-------------------+



In [0]:
spark.sql("""
select 
d.doctor_id,
d.doctor_name,
d.specialization,
d.city,
count(v.visit_id) as total_visits,
sum(v.bill_amount) as total_revenue
from doctors d
left join visits v
on d.doctor_id=v.doctor_id
group by 
d.doctor_id,
d.doctor_name,
d.specialization,
d.city
""").show()

+---------+-----------+--------------+---------+------------+-------------+
|doctor_id|doctor_name|specialization|     city|total_visits|total_revenue|
+---------+-----------+--------------+---------+------------+-------------+
|     D102|  Dr. Priya|     Neurology|Bangalore|           1|         3500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|           2|        18000|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|           1|         1500|
|     D107| Dr. Farhan|     Neurology|     Pune|           0|         NULL|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|           1|         7000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|           2|         2000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|           2|        10500|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|           0|         NULL|
+---------+-----------+--------------+---------+------------+-------------+



In [0]:
etl_doctors = spark.read.option(
    "header","true"
).option(
    "inferSchema","true"
).csv("dbfs:/FileStore/doctors.csv")

In [0]:
etl_visits = spark.read.option(
    "header","true"
).option(
    "inferSchema","true"
).csv("dbfs:/FileStore/visits.csv")

In [0]:
etl_hospital = spark.read.option(
    "multiline","true"
).json("dbfs:/FileStore/hospital_config.json")

In [0]:
etl_visits = etl_visits.fillna({
    "bill_amount":0
})

In [0]:
etl_hospital = etl_hospital.select(
    "hospital_id",
    "hospital_name",
    "city",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    "services"
)

In [0]:
etl_join = etl_doctors.join(
    etl_visits,
    "doctor_id",
    "left"
)

In [0]:
etl_join = etl_join.withColumn(
    "revenue",
    col("bill_amount")
)

In [0]:
doctor_rank = etl_join.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization"
).agg(
    sum("revenue").alias("total_revenue")
)

In [0]:
doctor_rank = doctor_rank.withColumn(
    "rank",
    rank().over(
        Window.orderBy(desc("total_revenue"))
    )
)


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
specialization_summary = etl_join.groupBy(
    "specialization"
).agg(
    count("visit_id").alias("total_visits"),
    sum("revenue").alias("total_revenue")
)

In [0]:
etl_join.write.mode(
    "overwrite"
).parquet("silver/hospital_operations")

In [0]:
doctor_rank.write.mode(
    "overwrite"
).parquet(
    "gold/doctor_ranking"
)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
specialization_summary.write.mode(
    "overwrite"
).parquet(
    "gold/specialization_summary"
)

In [0]:
dashboard_dataset = etl_join.select(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city",
    "patient_name",
    "visit_id",
    "bill_amount",
    "payment_",
    "revenue"
)
dashboard_dataset.write.mode(
    "overwrite"
).parquet("dbfs:/gold/gold/hospital_dashboard")

dashboard_dataset.show()

+---------+-----------+--------------+---------+------------+--------+-----------+--------+-------+
|doctor_id|doctor_name|specialization|     city|patient_name|visit_id|bill_amount|payment_|revenue|
+---------+-----------+--------------+---------+------------+--------+-----------+--------+-------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|  Farhan Ali|   V1005|       1500|    Paid|   1500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai| Sneha Patel|   V1004|      12000|    Paid|  12000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|   Kiran Rao|   V1009|       6000|    Paid|   6000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|  Amit Kumar|   V1003|       2000| Pending|   2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|  Meera Nair|   V1008|          0| Pending|      0|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|  Neha Singh|   V1006|       7000|    Paid|   7000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|Rahul Sharma|   V1001|       5000|    Paid|   5000|
